# Visual Learning session metadata

Builds `visual_learning_session_metadata.csv` — one row per session for the six
Visual Learning mice.

In [1]:
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
OUTPUT_DIR = '/data/metadata'
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   version="v2",
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

https://api.allenneuraldynamics.org/v2/metadata_index/data_assets


In [3]:
# The cohort is defined by subject: five different project_name values are
# interleaved across the same six mice.
VISUAL_LEARNING_MICE = ['782149', '790322', '788406', '800792', '800995', '804363']

# Processed asset names end in _processed_<date>_<time>. Anchoring at end-of-string
# drops further-derived assets (behavior-nwb, cortical-zstack, coreg, ROICat) that
# carry _processed_ mid-name.
PROCESSED_PATTERN = (r'^multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+'
                     r'_processed_\d{4}-\d{2}-\d{2}_[\d-]+$')

# The QC request sends one asset name per document, so the whole cohort at once
# exceeds the gateway's header limit -- fetch it in batches.
BATCH = 40

## Query

In [4]:
aggregate = [
  {
    "$match": {
      "data_description.subject_id": {"$in": VISUAL_LEARNING_MICE},
      "name": {"$regex": "^multiplane-ophys_"},
      # exclude the post-training passive block
      "acquisition.acquisition_type": {"$exists": True,
                                       "$ne": "CENTER_MOUSEMOTION"},
    },
  },
  {
    "$project": {
      "name": 1,
      "subject_id": "$data_description.subject_id",
      "project_name": "$data_description.project_name",
      "acquisition_type": "$acquisition.acquisition_type",
      "session_start_time": "$acquisition.acquisition_start_time",
      "session_end_time": "$acquisition.acquisition_end_time",
      "rig": "$acquisition.instrument_id",
      "genotype": "$subject.subject_details.genotype",
      "sex": "$subject.subject_details.sex",
      "date_of_birth": "$subject.subject_details.date_of_birth",
      # flatten data_streams[] -> configurations[] -> images[] -> planes[]
      "planes": {"$reduce": {
          "input": {"$reduce": {
              "input": "$acquisition.data_streams", "initialValue": [],
              "in": {"$concatArrays": [
                  "$$value", {"$ifNull": ["$$this.configurations", []]}]}}},
          "initialValue": [],
          "in": {"$concatArrays": ["$$value",
              {"$reduce": {
                  "input": {"$ifNull": ["$$this.images", []]}, "initialValue": [],
                  "in": {"$concatArrays": [
                      "$$value", {"$ifNull": ["$$this.planes", []]}]}}}]}}},
    }
  },
  {
    "$project": {
      "name": 1, "subject_id": 1, "project_name": 1, "acquisition_type": 1,
      "session_start_time": 1, "session_end_time": 1, "rig": 1,
      "genotype": 1, "sex": 1, "date_of_birth": 1,
      "n_planes": {"$size": "$planes"},
      "plane_indices": "$planes.plane_index",
      "imaging_depths": "$planes.depth",
      "targeted_structures": "$planes.targeted_structure.acronym",
    }
  },
  # drop the 2-plane test sessions (800792, 800995 -- 2 planes in every
  # processing generation, so nothing is recovered by keeping them)
  {"$match": {"n_planes": 8}},
]

records = docdb_api_client.aggregate_docdb_records(
    pipeline = aggregate,
)
print(f'{len(records)} assets')

if len(records) == 0:
    raise RuntimeError('No assets matched -- check the client version and pipeline.')

1787 assets


## Session table

One row per session, keeping only the newest `_processed_` generation: a session is
reprocessed whenever the pipeline changes, so it appears several times (3-8 deep) under
different stamps. The stamp format sorts lexicographically in chronological order, so
`sort_values` + `keep='last'` picks the newest.

In [5]:
sessions = pd.DataFrame(records)
sessions = sessions[sessions.name.str.match(PROCESSED_PATTERN)].copy()

sessions['session_id'] = sessions.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
sessions['processed_stamp'] = sessions.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

sessions = (sessions.sort_values('processed_stamp')
                    .drop_duplicates('session_id', keep='last'))

print(f'{len(sessions)} unique sessions across {sessions.subject_id.nunique()} mice')
print(sessions.subject_id.value_counts().sort_index().to_string())

147 unique sessions across 6 mice
subject_id
782149    24
788406    32
790322    24
800792    25
800995    22
804363    20


In [6]:
# acquisition_date comes off the asset name; session_date/time off the timestamp.
# They agree on every row today -- kept separate because the name is what the mount
# and every derived asset are keyed by.
sessions['acquisition_date'] = sessions.session_id.str.extract(r'_(\d{4}-\d{2}-\d{2})_')
sessions['session_date'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).date())
sessions['session_time'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).time())
sessions['date_of_birth'] = sessions.date_of_birth.map(
    lambda x: datetime.strptime(x, '%Y-%m-%d').date() if isinstance(x, str) else x)
sessions['age_days'] = [(a - b).days if pd.notnull(b) else np.nan
                        for a, b in zip(pd.to_datetime(sessions.acquisition_date).dt.date,
                                        sessions.date_of_birth)]

sessions['session_type'] = sessions.acquisition_type
sessions['stage'] = sessions.session_type.str.extract(
    r'^(TRAINING_\d|OPHYS_\d|STAGE_\d)')
sessions['image_set'] = sessions.session_type.str.extract(r'_images_([AB])')

sessions = sessions.sort_values(['subject_id', 'acquisition_date'])
sessions['session_number'] = sessions.groupby('subject_id').cumcount() + 1

# Plane columns, ordered by plane_index so depths line up with names
sessions['plane_names'] = [
    [f'{s}_{i}' for i, s in sorted(zip(r.plane_indices, r.targeted_structures))]
    for r in sessions.itertuples()]
sessions['imaging_depths'] = [
    [d for _, d in sorted(zip(r.plane_indices, r.imaging_depths))]
    for r in sessions.itertuples()]
sessions['targeted_structures'] = [
    sorted(set(r.targeted_structures)) for r in sessions.itertuples()]

## Z-drift QC

QC lives in `quality_control.metrics` — a flat array of per-plane metrics, each with a
`status_history` whose last entry is current. Metric names carry the plane either
leading (`VISp_0 Z-drift Analysis`) or trailing
(`VISp_0 Z-drift Analysis - VISp_0`), so we check both ends.

Sessions whose processing generation predates the z-drift evaluation have no metric to
read; those stay `NA` rather than `0`, so a session with no QC is not mistaken for a
session that passed.

In [7]:
zdrift = []
targets = sessions.name.tolist()

for i in range(0, len(targets), BATCH):
    docs = docdb_api_client.retrieve_docdb_records(
        filter_query={'name': {'$in': targets[i:i + BATCH]}},
        projection={'name': 1, 'quality_control.metrics': 1},
        limit=BATCH,
    )
    for doc in docs:
        for metric in ((doc.get('quality_control') or {}).get('metrics') or []):
            name = str(metric.get('name'))
            if not re.search(r'z-?drift', name, re.I):
                continue
            history = metric.get('status_history') or []
            zdrift.append({
                'name': doc['name'],
                'metric_name': name,
                'status': history[-1].get('status') if history else None,
            })

zdrift = pd.DataFrame(zdrift)
print(f'{len(zdrift)} z-drift metric rows from {zdrift.name.nunique()} assets')

# plane may lead or trail the metric name
zdrift['plane_name'] = zdrift.metric_name.str.extract(r'^(VISp_\d+)')[0].fillna(
    zdrift.metric_name.str.extract(r'(VISp_\d+)\s*$')[0])
assert zdrift.plane_name.notna().all(), 'unparsed plane in a z-drift metric name'

zdrift = zdrift.drop_duplicates(['name', 'plane_name'])
print(zdrift.status.value_counts().to_string())

904 z-drift metric rows from 113 assets
status
Pass    788
Fail    116


In [8]:
# Failing plane names per session, plus the count. Sessions with no z-drift QC
# stay NA -- distinct from an empty list, which means QC ran and nothing failed.
failed = zdrift[zdrift.status == 'Fail'].copy()
failed['plane_index'] = failed.plane_name.str.extract(r'_(\d+)$').astype(int)

# sort by plane index, not lexically (VISp_10 would otherwise precede VISp_2)
failed_names = (failed.sort_values(['name', 'plane_index'])
                      .groupby('name').plane_name.apply(list))

have_qc = sessions.name.isin(zdrift.name)

sessions['planes_failing_zdrift'] = [
    (failed_names.get(n, []) if has else pd.NA)
    for n, has in zip(sessions.name, have_qc)]
sessions['n_planes_failing_zdrift'] = (
    sessions.name.map(zdrift.status.eq('Fail').groupby(zdrift.name).sum())
            .where(have_qc).astype('Int64'))

print(f'{int(have_qc.sum())} sessions with z-drift QC, '
      f'{int((~have_qc).sum())} left NA')
print(sessions.n_planes_failing_zdrift.value_counts(dropna=False).sort_index().to_string())

113 sessions with z-drift QC, 34 left NA
n_planes_failing_zdrift
0       71
1       14
2       10
3        5
4        6
5        2
6        3
7        1
8        1
<NA>    34


In [9]:
order = ['subject_id', 'session_id', 'name', 'session_type', 'acquisition_type',
         'stage', 'image_set', 'session_number', 'acquisition_date', 'session_date',
         'session_time', 'age_days', 'genotype', 'sex', 'date_of_birth', 'rig',
         'project_name', 'n_planes', 'plane_names', 'imaging_depths',
         'targeted_structures', 'planes_failing_zdrift',
         'n_planes_failing_zdrift', 'processed_stamp', '_id']

sessions = sessions[order].reset_index(drop=True)
sessions

,subject_id,session_id,name,session_type,acquisition_type,stage,image_set,session_number,acquisition_date,session_date,session_time,age_days,genotype,sex,date_of_birth,rig,project_name,n_planes,plane_names,imaging_depths,targeted_structures,planes_failing_zdrift,n_planes_failing_zdrift,processed_stamp,_id
0,782149,multiplane-ophys_782149_2025-03-25_09-46-08,multiplane-ophys_782149_2025-03-25_09-46-08_pr...,TRAINING_0_gratings_autorewards_15min,TRAINING_0_gratings_autorewards_15min,TRAINING_0,NaN,1,2025-03-25,2025-03-25,09:46:08.591468,108,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,422_MESO2_20241017,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[40, 320, 80, 280, 120, 240, 160, 200]",[VISp],[],0,2026-08-19_00-32-51,aca6e6d2-9f33-4ed6-8a66-86d69a282c30
1,782149,multiplane-ophys_782149_2025-03-28_10-55-25,multiplane-ophys_782149_2025-03-28_10-55-25_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,2,2025-03-28,2025-03-28,10:55:25.569080,111,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 114, 244, 80, 280, 40, 310]",[VISp],[],0,2026-08-19_00-34-09,2a3f8254-9f86-42fd-b4d7-fe1877ebf959
2,782149,multiplane-ophys_782149_2025-03-29_10-10-29,multiplane-ophys_782149_2025-03-29_10-10-29_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,3,2025-03-29,2025-03-29,10:10:29.493070,112,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[158, 198, 114, 246, 80, 276, 43, 306]",[VISp],"[VISp_0, VISp_4, VISp_6]",3,2026-08-19_00-33-54,7591b588-f6a1-474d-b00a-3dd10ffb4a60
3,782149,multiplane-ophys_782149_2025-03-31_12-23-33,multiplane-ophys_782149_2025-03-31_12-23-33_pr...,TRAINING_1_gratings,TRAINING_1_gratings,TRAINING_1,NaN,4,2025-03-31,2025-03-31,12:23:33.753970,114,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 245, 80, 280, 40, 310]",[VISp],[],0,2026-08-19_00-34-28,d00bf70e-41c7-4ec5-ac76-8aecb010fbe1
4,782149,multiplane-ophys_782149_2025-04-01_09-42-11,multiplane-ophys_782149_2025-04-01_09-42-11_pr...,TRAINING_2_gratings_flashed,TRAINING_2_gratings_flashed,TRAINING_2,NaN,5,2025-04-01,2025-04-01,09:42:11.814685,115,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Male,2024-12-07,429_MESO1_20241016,LearningmFISHTask1A,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 115, 240, 85, 280, 40, 300]",[VISp],"[VISp_4, VISp_6]",2,2026-08-19_00-33-58,ac0f8e4d-cd12-453a-8d83-6b7951a6a957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,804363,multiplane-ophys_804363_2025-09-04_16-08-45,multiplane-ophys_804363_2025-09-04_16-08-45_pr...,OPHYS_6_images_B,OPHYS_6_images_B,OPHYS_6,B,16,2025-09-04,2025-09-04,16:08:45.114554,130,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Female,2025-04-27,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 200, 120, 238, 80, 272, 48, 314]",[VISp],"[VISp_0, VISp_1, VISp_2, VISp_4, VISp_5, VISp_6]",6,2026-08-19_01-05-08,a7ae1f04-6d75-4f43-ada6-61055ef90896
143,804363,multiplane-ophys_804363_2025-09-05_13-25-33,multiplane-ophys_804363_2025-09-05_13-25-33_pr...,STAGE_0,STAGE_0,STAGE_0,NaN,17,2025-09-05,2025-09-05,13:25:33.172779,131,Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-IC...,Female,2025-04-27,429_MESO1_20241016,Learning mFISH-V1omFISH,8,"[VISp_0, VISp_1, VISp_2, VISp_3, VISp_4, VISp_...","[160, 196, 120, 240, 78, 276, 44, 320]",[VISp],<NA>,<NA>,2026-08-19_01-05-09,100f82f1-b73a-49c2-998d-dbcabf9362e7
144,804363,multiplane-ophys_804363_2025-09-08_09-24-39,multiplane-ophys_804363_2025-09-08_09-24-39_pr...,STAGE_1,STAGE_1,STAGE_1,NaN,18,2025-09-08,2025-09-08,09:24

## Sanity checks

docDB drops rows silently — it returns no error when an asset simply is not indexed.
Read these counts against what you expect from the processing batch; if a mouse is
short, re-run rather than assuming the data is missing.

In [10]:
print('sessions per mouse')
print(sessions.subject_id.value_counts().sort_index().to_string())

print('\nplanes per session (8 for all -- enforced in the query)')
print(sessions.n_planes.value_counts().sort_index().to_string())

print('\nsession types')
print(sessions.session_type.value_counts().to_string())

missing = set(VISUAL_LEARNING_MICE) - set(sessions.subject_id)
if missing:
    print(f'\nno sessions returned for: {sorted(missing)}')

sessions per mouse
subject_id
782149    24
788406    32
790322    24
800792    25
800995    22
804363    20

planes per session (8 for all -- enforced in the query)
n_planes
8    147

session types
session_type
TRAINING_1_gratings                      26
TRAINING_3_images_A_10uL_reward          19
STAGE_1                                  19
OPHYS_6_images_B                         15
OPHYS_1_images_A                         12
OPHYS_4_images_B                         12
TRAINING_2_gratings_flashed               9
TRAINING_4_images_A_training              7
TRAINING_0_gratings_autorewards_15min     7
TRAINING_5_images_A_epilogue              7
STAGE_0                                   7
TRAINING_5_images_A_handoff_ready         6
TRAINING_5_images_A_handoff_lapsed        1


## Write the CSV

In [11]:
session_csv = f'{OUTPUT_DIR}/visual_learning_session_metadata.csv'
sessions.to_csv(session_csv, index=False)
print(f'{session_csv}  ({len(sessions)} rows, {sessions.shape[1]} columns)')

/data/metadata/visual_learning_session_metadata.csv  (147 rows, 25 columns)


---

## What's in this table?

Use this to see what the dataset actually offers before picking sessions for a
problem set.

In [12]:
# Column inventory: type, fill rate, and how much each column varies
rows = []
for col in sessions.columns:
    s = sessions[col]
    as_str = s.map(lambda v: str(v) if isinstance(v, list) else v)
    rows.append({
        'column': col,
        'dtype': str(s.dtype),
        'n_missing': int(s.isna().sum()),
        'n_unique': int(as_str.nunique(dropna=True)),
        'example': str(s.dropna().iloc[0])[:44] if s.notna().any() else '',
    })
pd.DataFrame(rows)

,column,dtype,n_missing,n_unique,example
0,subject_id,object,0,6,782149
1,session_id,object,0,147,multiplane-ophys_782149_2025-03-25_09-46-08
2,name,object,0,147,multiplane-ophys_782149_2025-03-25_09-46-08_
3,session_type,object,0,13,TRAINING_0_gratings_autorewards_15min
4,acquisition_type,object,0,13,TRAINING_0_gratings_autorewards_15min
5,stage,object,0,11,TRAINING_0
6,image_set,object,68,2,A
7,session_number,int64,0,32,1
8,acquisition_date,object,0,96,2025-03-25
9,session_date,object,0,96,2025-03-25


In [13]:
# Which columns are constant across the cohort (no use as a selector)?
varying, constant = [], []
for col in sessions.columns:
    as_str = sessions[col].map(lambda v: str(v) if isinstance(v, list) else v)
    (constant if as_str.nunique(dropna=True) <= 1 else varying).append(col)

print(f'constant across all {len(sessions)} sessions:')
for c in constant:
    print(f'  {c} = {sessions[c].dropna().iloc[0] if sessions[c].notna().any() else "all NA"}')
print(f'\nvarying ({len(varying)}): {varying}')

constant across all 147 sessions:
  genotype = Slc32a1-IRES-Cre/wt;Oi1(TIT2L-jGCaMP8s-WPRE-ICL-IRES-tTA2)/wt
  n_planes = 8
  plane_names = ['VISp_0', 'VISp_1', 'VISp_2', 'VISp_3', 'VISp_4', 'VISp_5', 'VISp_6', 'VISp_7']
  targeted_structures = ['VISp']

varying (21): ['subject_id', 'session_id', 'name', 'session_type', 'acquisition_type', 'stage', 'image_set', 'session_number', 'acquisition_date', 'session_date', 'session_time', 'age_days', 'sex', 'date_of_birth', 'rig', 'project_name', 'imaging_depths', 'planes_failing_zdrift', 'n_planes_failing_zdrift', 'processed_stamp', '_id']


In [14]:
# Categorical columns worth filtering on
for col in ['session_type', 'image_set', 'rig', 'project_name', 'sex']:
    counts = sessions[col].value_counts(dropna=False)
    print(f'--- {col} ({counts.size} values)')
    print(counts.to_string(), '\n')

--- session_type (13 values)
session_type
TRAINING_1_gratings                      26
TRAINING_3_images_A_10uL_reward          19
STAGE_1                                  19
OPHYS_6_images_B                         15
OPHYS_1_images_A                         12
OPHYS_4_images_B                         12
TRAINING_2_gratings_flashed               9
TRAINING_4_images_A_training              7
TRAINING_0_gratings_autorewards_15min     7
TRAINING_5_images_A_epilogue              7
STAGE_0                                   7
TRAINING_5_images_A_handoff_ready         6
TRAINING_5_images_A_handoff_lapsed        1 

--- image_set (3 values)
image_set
NaN    68
A      52
B      27 

--- rig (3 values)
rig
422_MESO2_20241017    75
429_MESO1_20241016    44
422_MESO2_20220218    28 

--- project_name (4 values)
project_name
Learning mFISH-V1omFISH    80
LearningmFISHTask1A        56
U01BFCT                     9
ISIx                        2 

--- sex (2 values)
sex
Male      80
Female    67 



In [15]:
# Sessions per mouse per session_type -- where the usable data actually is
pd.crosstab(sessions.subject_id, sessions.session_type,
            margins=True, margins_name='TOTAL')

session_type,OPHYS_1_images_A,OPHYS_4_images_B,OPHYS_6_images_B,STAGE_0,STAGE_1,TRAINING_0_gratings_autorewards_15min,TRAINING_1_gratings,TRAINING_2_gratings_flashed,TRAINING_3_images_A_10uL_reward,TRAINING_4_images_A_training,TRAINING_5_images_A_epilogue,TRAINING_5_images_A_handoff_lapsed,TRAINING_5_images_A_handoff_ready,TOTAL
subject_id,,,,,,,,,,,,,,
782149,2,2,2,1,3,1,3,1,3,2,2,1,1,24
788406,2,2,3,2,3,1,11,2,3,1,1,0,1,32
790322,2,2,2,1,4,2,3,2,3,1,1,0,1,24
800792,2,2,3,1,3,1,4,2,4,1,1,0,1,25
800995,2,2,3,1,3,1,3,1,3,1,1,0,1,22
804363,2,2,2,1,3,1,2,1,3,1,1,0,1,20
TOTAL,12,12,15,7,19,7,26,9,19,7,7,1,6,147


In [16]:
# Imaging geometry: are the 8 planes at consistent depths across sessions?
depths = sessions.explode('imaging_depths')
print('distinct imaging depths:', sorted(depths.imaging_depths.dropna().unique()))
print('\ndepth range per plane count')
print(sessions.groupby('n_planes').imaging_depths.apply(
    lambda col: f'{min(min(d) for d in col)} - {max(max(d) for d in col)} um').to_string())

print('\ntargeted structures:',
      sorted({s for lst in sessions.targeted_structures for s in lst}))

distinct imaging depths: [20, 22, 26, 28, 29, 30, 32, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 48, 50, 64, 66, 68, 70, 72, 74, 75, 76, 78, 80, 81, 82, 84, 85, 86, 88, 90, 92, 94, 95, 96, 98, 100, 102, 104, 105, 108, 110, 112, 114, 115, 116, 118, 120, 122, 124, 126, 127, 128, 129, 130, 132, 139, 144, 146, 148, 150, 152, 154, 155, 156, 157, 158, 160, 162, 164, 166, 167, 168, 170, 172, 174, 175, 179, 182, 186, 188, 190, 192, 194, 195, 196, 198, 200, 202, 204, 206, 208, 210, 212, 216, 219, 220, 222, 223, 224, 225, 226, 228, 230, 232, 234, 235, 236, 238, 240, 242, 244, 245, 246, 248, 250, 252, 254, 256, 257, 258, 260, 262, 263, 264, 265, 266, 268, 270, 272, 274, 275, 276, 278, 280, 282, 284, 288, 289, 290, 292, 294, 296, 298, 300, 302, 304, 305, 306, 308, 310, 312, 314, 316, 318, 320, 322, 324, 328, 330, 344, 345, 347, 348, 354]

depth range per plane count
n_planes
8    20 - 354 um

targeted structures: ['VISp']


In [17]:
# Numeric spread, and how age and session count relate per mouse
print(sessions[['age_days', 'n_planes', 'session_number',
                'n_planes_failing_zdrift']].describe().to_string())

print('\nper-mouse span')
print(sessions.groupby('subject_id').agg(
    n_sessions=('session_id', 'size'),
    first_date=('acquisition_date', 'min'),
    last_date=('acquisition_date', 'max'),
    age_first=('age_days', 'min'),
    age_last=('age_days', 'max'),
).to_string())

         age_days  n_planes  session_number  n_planes_failing_zdrift
count  147.000000     147.0      147.000000                    113.0
mean   141.020408       8.0       13.034014                 1.026549
std     22.161138       0.0        7.610748                 1.759972
min    107.000000       8.0        1.000000                      0.0
25%    124.500000       8.0        7.000000                      0.0
50%    137.000000       8.0       13.000000                      0.0
75%    151.500000       8.0       19.000000                      1.0
max    202.000000       8.0       32.000000                      8.0

per-mouse span
            n_sessions  first_date   last_date  age_first  age_last
subject_id                                                         
782149              24  2025-03-25  2025-05-07        108       151
788406              32  2025-05-29  2025-07-29        131       192
790322              24  2025-06-11  2025-08-21        131       202
800792              25 

In [18]:
# Z-drift QC coverage -- and the caveat that NA is not a pass
qc_cov = sessions.n_planes_failing_zdrift.notna()
print(f'sessions with z-drift QC: {int(qc_cov.sum())} / {len(sessions)}')
print(f'  clean (0 failing planes):  {int((sessions.n_planes_failing_zdrift == 0).sum())}')
print(f'  >=1 failing plane:         {int((sessions.n_planes_failing_zdrift > 0).sum())}')
print(f'  no QC (NA, NOT a pass):    {int((~qc_cov).sum())}')

print('\nQC coverage by session_type')
print(sessions.assign(has_qc=qc_cov).groupby('session_type').has_qc.agg(
    n='size', with_qc='sum').to_string())

sessions with z-drift QC: 113 / 147
  clean (0 failing planes):  71
  >=1 failing plane:         42
  no QC (NA, NOT a pass):    34

QC coverage by session_type
                                        n  with_qc
session_type                                      
OPHYS_1_images_A                       12       12
OPHYS_4_images_B                       12       12
OPHYS_6_images_B                       15       15
STAGE_0                                 7        5
STAGE_1                                19       14
TRAINING_0_gratings_autorewards_15min   7        3
TRAINING_1_gratings                    26       17
TRAINING_2_gratings_flashed             9        5
TRAINING_3_images_A_10uL_reward        19       11
TRAINING_4_images_A_training            7        5
TRAINING_5_images_A_epilogue            7        7
TRAINING_5_images_A_handoff_lapsed      1        1
TRAINING_5_images_A_handoff_ready       6        6


In [19]:
# Candidate sessions for a problem set: QC present and nothing failing
usable = sessions[sessions.n_planes_failing_zdrift == 0]
print(f'{len(usable)} sessions with zero z-drift failures')
print(usable.groupby(['subject_id', 'session_type']).size().to_string())

# Which planes fail z-drift most often across the cohort?
exploded = sessions.planes_failing_zdrift.dropna().explode().dropna()
print('\nz-drift failures by plane')
print(exploded.value_counts().sort_index().to_string())

71 sessions with zero z-drift failures
subject_id  session_type                         
782149      OPHYS_4_images_B                         1
            OPHYS_6_images_B                         1
            STAGE_0                                  1
            TRAINING_0_gratings_autorewards_15min    1
            TRAINING_1_gratings                      2
            TRAINING_3_images_A_10uL_reward          1
            TRAINING_5_images_A_epilogue             1
788406      STAGE_1                                  1
            TRAINING_0_gratings_autorewards_15min    1
            TRAINING_1_gratings                      6
            TRAINING_2_gratings_flashed              2
            TRAINING_3_images_A_10uL_reward          3
            TRAINING_4_images_A_training             1
            TRAINING_5_images_A_handoff_ready        1
790322      OPHYS_1_images_A                         2
            OPHYS_4_images_B                         2
            OPHYS_6_images_B   